In [ ]:
from playwright.sync_api import sync_playwright

SOURCE_URL = "https://hp.cbg.163.com/cgi/mweb/pl?search_type=role&tfid=f_kingkong&collect_all_cards=1&refer_sn=01A01E0E-E604-0AEE-AFB6-8CD1548136B7"

def fetch_card_content(url=SOURCE_URL):
    """优先选择卡牌数量最多的账号，返回内存中的 .content，不保存 HTML。"""
    with sync_playwright() as p:
        browser = p.chromium.launch(channel="chrome", headless=False)
        try:
            page = browser.new_page(viewport={"width": 1440, "height": 1000})
            best_html = ""
            best_count = -1
            page.goto(url, wait_until="domcontentloaded", timeout=120000)
            page.locator(".product-item.product-item-cards").first.wait_for(state="attached", timeout=120000)
            candidate_total = page.locator(".product-item.product-item-cards").count()
            for candidate_index in range(candidate_total):
                if candidate_index > 0:
                    page.goto(url, wait_until="domcontentloaded", timeout=120000)
                    page.locator(".product-item.product-item-cards").first.wait_for(state="attached", timeout=120000)
                page.locator(".product-item.product-item-cards").nth(candidate_index).click(timeout=120000)
                page.locator(".tabs .item").filter(has_text="卡牌").first.click(timeout=120000)
                content = page.locator(".content").first
                content.wait_for(state="attached", timeout=120000)
                page.wait_for_timeout(500)
                for _ in range(12):
                    page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    page.wait_for_timeout(300)
                page.evaluate("window.scrollTo(0, 0)")
                count = content.locator("li.ico-wrap.clickable").count()
                if count > best_count:
                    best_count = count
                    best_html = content.evaluate("element => element.outerHTML")
                if count >= 141:
                    break
            if not best_html:
                raise RuntimeError("没有找到账号卡牌内容")
            print(f"选择卡牌数量最多的账号：{best_count} 张")
            return best_html
        finally:
            browser.close()

html = fetch_card_content()
print(f"已获取网页内容：{len(html):,} 个字符（未保存为本地 HTML）")

In [ ]:
from bs4 import BeautifulSoup
from requests import get
import json
import re
from html import unescape
from pathlib import Path
from urllib.parse import urlparse

if "html" not in globals():
    raise RuntimeError("请先运行第一个单元格获取网页内容")

shuliandu = "https://cbg-hp.res.netease.com/game_res/unpack_res/cbg/others/pic_shuliandu_huo_01.png"
pinzhi_map = {"01": "普通", "02": "稀有", "03": "史诗", "04": "传说", "05": "魔法", "06": "神话"}

soup = BeautifulSoup(html, "html.parser")
card_nodes = soup.select("li.ico-wrap.clickable")
cards = {}
for index, node in enumerate(card_nodes):
    name_node = node.select_one(".cqm-text.icon-label")
    card_img = node.select_one('img[src*="/skill_card_small/"]')
    frame_img = node.select_one('img[src*="/pic_"][src*="kuang_"]')
    if not (name_node and card_img and frame_img):
        continue
    name = unescape(re.sub(r"\s+", " ", name_node.get_text(" ", strip=True))).strip()
    card_url = card_img.get("src", "")
    kuang_url = frame_img.get("src", "")
    frame_match = re.search(r"pic_(zhuchang|huoban|fashu)kuang_(\d+)\.png", kuang_url)
    if not frame_match:
        continue
    leixing = {"zhuchang": "召唤卡", "huoban": "伙伴卡", "fashu": "咒语卡"}[frame_match.group(1)]
    pinzhi = pinzhi_map.get(frame_match.group(2))
    if pinzhi is None:
        continue
    num_img = node.select_one('img[src*="pic_card_num_b"]')
    num_match = re.search(r"pic_card_num_b(\d+)\.png", num_img.get("src", "") if num_img else "")
    mastery_img = node.select_one('img[src*="pic_shuliandu"]')
    asset_name = Path(urlparse(card_url).path).stem
    cards[f"{index:03d}_{asset_name}"] = {
        "显示名称": name, "顺序": index, "素材名": asset_name,
        "魔力消耗": int(num_match.group(1)) if num_match else "",
        "类型": leixing, "品质": pinzhi, "卡片url": card_url, "框url": kuang_url,
        "魔力url": num_img.get("src", "") if num_img else "",
        "熟练度url": mastery_img.get("src", "") if mastery_img else "",
    }
print(f"按原网页顺序解析到 {len(cards)} 张卡牌")


解析到 141 张卡牌


In [ ]:
import json
# 原网页 DOM 顺序就是展示顺序，不做中文排序
sorted_data = dict(cards)
json.dump(sorted_data, open("卡牌信息.json", "w", encoding="utf-8"), ensure_ascii=False, indent=4)

In [ ]:
import os
def download(url, name):
    if os.path.exists(name) and os.path.getsize(name) > 0:
        return
    
    with open(name, "wb") as f:
        f.write(get(url).content)
        print(f"下载 {name} 完成")
    return name

In [ ]:
download(shuliandu, "background/shuliandu.png")

for key, card in cards.items():
    asset_name = card["素材名"]
    num = card["魔力消耗"]
    leixing = card["类型"]
    pinzhi = card["品质"]
    download(card["框url"], f"background/{leixing}_{pinzhi}.png")
    if card["魔力url"]:
        download(card["魔力url"], f"background/card_num_{num}.png")
    download(card["卡片url"], f"background/{asset_name}.png")

下载 background/福灵剂.png 完成
下载 background/纽特·斯卡曼德.png 完成
下载 background/隐形衣.png 完成
下载 background/风雨雷灾.png 完成
下载 background/龙守仁.png 完成
下载 background/蝙蝠袭来.png 完成
下载 background/护树罗锅.png 完成
下载 background/芭蕾巨怪.png 完成
下载 background/皮皮鬼.png 完成
下载 background/四分五裂.png 完成
下载 background/飞鸟群群.png 完成
下载 background/骑士公共汽车.png 完成
下载 background/热流咒.png 完成
下载 background/统统石化.png 完成
下载 background/飞路网.png 完成
下载 background/快快复苏.png 完成
下载 background/伙伴卡_普通.png 完成
下载 background/多洛雷斯·乌姆里奇.png 完成


In [ ]:
from html import escape
from pathlib import Path
from urllib.parse import quote

background_dir = Path("background")
output_path = Path("卡牌合成.html")
card_data = globals().get("sorted_data", cards)
mastery_path = background_dir / "shuliandu.png"

def asset_url(path):
    return quote(path.as_posix(), safe="/")

tiles = []
missing = []
for key, card in card_data.items():
    display_name = card["显示名称"]
    card_path = background_dir / f"{card['素材名']}.png"
    frame_path = background_dir / f"{card['类型']}_{card['品质']}.png"
    cost = card.get("魔力消耗", "")
    cost_path = background_dir / f"card_num_{cost}.png" if cost != "" else None
    if not card_path.exists() or not frame_path.exists():
        missing.append(display_name)
        continue
    mastery_html = f'<img class="card-mastery" src="{asset_url(mastery_path)}" alt="">' if cost != "" and mastery_path.exists() else ""
    cost_html = f'<img class="card-cost" src="{asset_url(cost_path)}" alt="">' if cost != "" and cost_path and cost_path.exists() else ""
    tiles.append(f'''<article class="card-item" data-type="{escape(card["类型"])}">
        <div class="card-visual">
          <img class="card-art" src="{asset_url(card_path)}" alt="{escape(display_name)}">
          <img class="card-frame" src="{asset_url(frame_path)}" alt="">
          {mastery_html}{cost_html}
        </div>
        <span class="card-name">{escape(display_name)}</span>
      </article>''')

filter_buttons = ['<button class="filter on" data-filter="全部">全部</button>']
for card_type in ("咒语卡", "召唤卡", "伙伴卡"):
    filter_buttons.append(f'<button class="filter" data-filter="{card_type}">{card_type}</button>')

document = '''<!doctype html>
<html lang="zh-CN"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>卡牌收集</title><style>
*{box-sizing:border-box}html{font-size:19.6px}body{margin:0;background:#f5f5f5;color:#1e1e1e;font-family:"PingFang SC","Microsoft Yahei","Heiti SC",miui,helvetica,sans-serif}
.page{width:750px;margin:0 auto;background:#fff;min-height:100vh}.tabs{display:flex;height:67.2px;padding-left:14px;background:#fff}
.filter{display:block;height:39.2px;margin:14px 14px 14px 0;padding:0 42px;border:0;background:#f5f5f5;color:#1e1e1e;font:inherit;font-size:16.8px;line-height:36.4px;cursor:pointer}.filter.on{background:#fff;color:#ff3a36}
.collection-title{position:relative;height:50.4px;padding:15.4px 14px 11.2px 25.2px;background:#fff;font-size:19.6px;font-weight:700;line-height:19.6px}.collection-title:before{content:"";position:absolute;left:14px;top:11.2px;width:4px;height:28px;border-radius:2px;background:#ff3a36}
.card-section{padding:0 14px 11.2px;background:#fff}.card-grid{display:grid;grid-template-columns:repeat(5,1fr);width:100%}.card-item{width:100%;min-width:0;margin:0 0 11.2px;padding:0 11.2px 0 0}.card-visual{position:relative;width:100%;aspect-ratio:135.438/194.688;margin:0 0 7px;overflow:visible}.card-art{position:absolute;z-index:1;left:4%;top:10%;width:92%;height:auto}.card-frame{position:absolute;z-index:2;inset:0;width:100%;height:auto}.card-mastery,.card-cost{position:absolute;z-index:3;height:auto}.card-mastery{left:-.2rem;top:0rem;width:.9rem}.card-cost{left:-.1rem;top:.65rem;width:.7rem}.card-name{display:flow-root;width:100%;min-height:42px;color:#1e1e1e;font-size:16.8px;line-height:21px;text-align:center;overflow-wrap:anywhere}@media(max-width:750px){.page{width:100%}}
</style></head><body><main class="page"><nav class="tabs">''' + "".join(filter_buttons) + '''</nav>
<div class="collection-title">卡牌收集进度：''' + str(len(tiles)) + "/" + str(len(tiles)) + '''</div><section class="card-section"><div class="card-grid">''' + "".join(tiles) + '''</div></section></main>
<script>document.querySelectorAll(".filter").forEach(button=>button.addEventListener("click",()=>{document.querySelectorAll(".filter").forEach(item=>item.classList.remove("on"));button.classList.add("on");const filter=button.dataset.filter;document.querySelectorAll(".card-item").forEach(card=>{card.hidden=filter!=="全部"&&card.dataset.type!==filter})}))</script></body></html>'''

output_path.write_text(document, encoding="utf-8")
print(f"已生成：{output_path.resolve()}")
print(f"已合成 {len(tiles)} 张卡牌；缺少素材 {len(missing)} 张")
if missing:
    print("缺少素材：", ", ".join(missing[:20]))
